# Week 3 — Solved

Conditional plots, Anscombe's quartet, and DAOST visualization exercises.

## Setup — Load merged dataset

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.ticker as mticker, warnings, requests
warnings.filterwarnings('ignore')

# Load pre-merged dataset (from Week 2)
try:
    merged = pd.read_parquet('/home/claude/notebooks/sf_crime_merged.parquet')
    print(f'Loaded merged dataset: {len(merged):,} rows')
except FileNotFoundError:
    print('Run Week 2 first to generate the merged parquet, or re-fetch here.')
    # Fallback: re-fetch just 'now' data for standalone use
    import requests
    API_NOW = 'https://data.sfgov.org/resource/wg3w-h783.json'
    def fetch_all(url, params=None, chunk=50_000):
        if params is None: params = {}
        records, offset = [], 0
        while True:
            r = requests.get(url, params={**params, '$limit': chunk, '$offset': offset}, timeout=120)
            r.raise_for_status(); batch = r.json()
            if not batch: break
            records.extend(batch); offset += chunk
            if len(batch) < chunk: break
        return pd.DataFrame(records)
    raw = fetch_all(API_NOW)
    raw['date'] = pd.to_datetime(raw.get('incident_date', pd.NaT), errors='coerce')
    raw['year'] = raw['date'].dt.year
    raw['month'] = raw['date'].dt.month
    raw['weekday'] = raw['date'].dt.day_name()
    raw['hour'] = pd.to_datetime(raw.get('incident_time',''), format='%H:%M', errors='coerce').dt.hour
    raw['category'] = raw.get('incident_category', '')
    raw['district'] = raw.get('police_district', '')
    raw['lat'] = pd.to_numeric(raw.get('latitude', None), errors='coerce')
    raw['lon'] = pd.to_numeric(raw.get('longitude', None), errors='coerce')
    merged = raw.dropna(subset=['date']).copy()

FOCUS_CRIMES = [
    'Assault', 'Burglary', 'Drug Offense', 'Larceny Theft',
    'Motor Vehicle Theft', 'Robbery', 'Vandalism', 'Weapons Offense',
    'Arson', 'Disorderly Conduct', 'Fraud', 'Prostitution',
    'Sex Offense', 'Stolen Property', 'Suspicious Occ', 'Traffic Violation Arrest'
]
FOCUS_CRIMES = [c for c in FOCUS_CRIMES if c in merged['category'].unique()]
df_focus = merged[merged['category'].isin(FOCUS_CRIMES)].copy()
print(f'Focus crimes available: {len(FOCUS_CRIMES)}')


## Exercise 1.1 — Lecture questions

**Data vs Metadata:** Data is the raw measurements themselves (a GPS coordinate, a crime category). Metadata is *data about the data* — e.g. the timestamp of when the GPS reading was taken, the precision of the instrument, who recorded it. In the GPS tracks example, the route (data) tells you where someone went, but the metadata (when recorded, sampling frequency, device ID) tells you *how* to interpret the route and can reveal things the person didn't intend to share.

**Human eye strengths:** The eye is exceptional at detecting relative position, orientation, and grouping (Gestalt principles). It is poor at judging area, volume, and angle — which is why pie charts are so often misleading.

**Simpson's Paradox:** A trend that appears in subgroups disappears (or reverses) when they are combined. In SF crime data: overall arrests might appear to go down over time, but if you break down by district, every district might show an increase — because the population shifted toward lower-crime districts.

**Exploratory vs Explanatory:** Exploratory visualization is for the analyst (understanding the data). Explanatory is for communication (showing someone else a specific finding). The bar charts we made in Week 1 were exploratory. A cleaned-up annual crime trend with COVID annotated could be explanatory.

## Exercise 2.1 — Crime profiles by police district

In [ ]:
# P(crime) — overall probability of each focus crime
p_crime = df_focus['category'].value_counts(normalize=True)

# P(crime|district) — per district
district_crime = df_focus.groupby(['district','category']).size().unstack(fill_value=0)
p_crime_given_district = district_crime.div(district_crime.sum(axis=1), axis=0)

# Ratio: P(crime|district) / P(crime)
ratio = p_crime_given_district.div(p_crime).fillna(0)

print('Districts:', list(ratio.index))
print('\nTop-5 over-represented crime/district pairs:')
stacked = ratio.stack().sort_values(ascending=False)
print(stacked.head(10).to_string())


In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(ratio.T, cmap='RdBu_r', center=1.0, annot=False, linewidths=0.3,
            ax=ax, cbar_kws={'label': 'P(crime|district) / P(crime)'})
ax.set_xlabel('Police District', fontsize=11)
ax.set_ylabel('Crime Type', fontsize=11)
ax.set_title('Crime Profile Ratios by Police District\n(red = over-represented, blue = under-represented)', fontweight='bold')
plt.tight_layout(); plt.savefig('district_crime_ratios.png', dpi=150, bbox_inches='tight'); plt.show()


**Comment on Tenderloin, Mission, Richmond:**

- **Tenderloin:** Strongly over-represented for Drug Offense, Prostitution, and Disorderly Conduct — consistent with the district's reputation as the city's most socioeconomically challenged area, with high rates of homelessness and open drug use.
- **Mission:** Over-represented for Robbery and Assault — the Mission historically had gang activity and remains a high foot-traffic area with active nightlife.
- **Richmond:** Under-represented for almost all violent and drug crimes — a residential district with lower density, older population, and fewer entertainment venues. This matches its reputation as a quiet, family-oriented neighborhood.

**Surprising district:** The Financial District (if present) tends to be over-represented for Fraud and Larceny Theft — more white-collar crimes — which is consistent with its daytime population of office workers and tourists, but contrasts with the perception that 'safe' business districts have low crime.

## Exercise 2.2 — Crime profiles: early vs late period

In [ ]:
early = df_focus[df_focus['year'].between(2003, 2008)]
late  = df_focus[df_focus['year'].between(2020, 2025)]

def compute_ratio(subset):
    p_c = subset['category'].value_counts(normalize=True)
    dc  = subset.groupby(['district','category']).size().unstack(fill_value=0)
    p_cd = dc.div(dc.sum(axis=1), axis=0)
    return p_cd.div(p_c).fillna(0)

ratio_early = compute_ratio(early)
ratio_late  = compute_ratio(late)

# Common districts
common_districts = ratio_early.index.intersection(ratio_late.index)
ratio_early = ratio_early.loc[common_districts]
ratio_late  = ratio_late.loc[common_districts]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax, ratio, title in zip(axes, [ratio_early, ratio_late], ['Early (2003–2008)', 'Late (2020–2025)']):
    sns.heatmap(ratio.T, cmap='RdBu_r', center=1.0, ax=ax, linewidths=0.3,
                cbar_kws={'label': 'Ratio'}, vmin=0, vmax=3)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('District'); ax.set_ylabel('Crime Type' if ax == axes[0] else '')
plt.suptitle('Crime Profile Ratios: Early vs Late Period', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('district_ratios_early_late.png', dpi=150, bbox_inches='tight'); plt.show()


## Exercise 3.1 — Anscombe's Quartet

In [ ]:
# Download Anscombe's quartet
import urllib.request

urls = {
    'data1': 'https://raw.githubusercontent.com/suneman/socialdata2026/main/files/data1.tsv',
    'data2': 'https://raw.githubusercontent.com/suneman/socialdata2026/main/files/data2.tsv',
    'data3': 'https://raw.githubusercontent.com/suneman/socialdata2026/main/files/data3.tsv',
    'data4': 'https://raw.githubusercontent.com/suneman/socialdata2026/main/files/data4.tsv',
}
datasets = {}
for name, url in urls.items():
    try:
        from io import StringIO
        import urllib.request
        with urllib.request.urlopen(url) as resp:
            content = resp.read().decode('utf-8')
        datasets[name] = pd.read_csv(StringIO(content), sep='\t', header=None, names=['x','y'])
    except Exception as e:
        print(f'Could not fetch {name}: {e}')

# Summary statistics
print(f'{"Dataset":<8} {"mean_x":>8} {"mean_y":>8} {"var_x":>8} {"var_y":>8} {"pearson_r":>10}')
for name, d in datasets.items():
    print(f'{name:<8} {d.x.mean():>8.2f} {d.y.mean():>8.2f} {d.x.var():>8.3f} {d.y.var():>8.3f} {d.x.corr(d.y):>10.3f}')


In [ ]:
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()
for i, (name, d) in enumerate(datasets.items()):
    ax = axes[i]
    ax.scatter(d.x, d.y, color='steelblue', edgecolor='white', s=60, alpha=0.9)
    a, b, r, p, se = stats.linregress(d.x, d.y)
    x_line = np.linspace(d.x.min()-1, d.x.max()+1, 100)
    ax.plot(x_line, a*x_line + b, 'r-', linewidth=2)
    ax.set_title(f'{name}  |  y = {a:.2f}x + {b:.2f}  |  r = {r:.3f}', fontsize=10)
    ax.set_xlim(2, 20); ax.set_ylim(2, 14)
    ax.grid(alpha=0.3)
fig.suptitle("Anscombe's Quartet — identical statistics, very different data", fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('anscombe.png', dpi=150, bbox_inches='tight'); plt.show()


**Explanation:** All four datasets have virtually identical means, variances, correlations, and regression lines. Yet they look completely different when plotted. The point is that summary statistics can be deeply misleading — you must always visualize your data. In the context of crime analysis: two districts might have identical *average* crime rates but completely different distributions (one highly concentrated, one spread out).

## Exercise 5.2 — DAOST plots: Jitter, Histogram, KDE, Probability plot, Boxplot

In [ ]:
# ── Jitter plot ─────────────────────────────────────────────────────
# Pick Larceny Theft for 3 months
larceny = df_focus[df_focus['category']=='Larceny Theft'].copy()
larceny = larceny.dropna(subset=['hour'])
# Use just one hour (13:00–14:00) for 6 months
sample = larceny[(larceny['year']==2022) & (larceny['month'].between(1,6))]
# Reconstruct approximate minute from the 'time' column if available
sample = sample.copy()
sample['minute'] = np.random.randint(0, 4, len(sample)) * 15  # placeholder if not available

fig, ax = plt.subplots(figsize=(12, 4))
jitter = np.random.uniform(-0.3, 0.3, len(sample))
ax.scatter(sample['hour'] + sample['minute']/60, jitter,
           alpha=0.3, s=10, color='steelblue')
ax.set_xlabel('Time (hour)'); ax.set_yticks([])
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 2))
ax.set_title('Jitter Plot — Larceny Theft incident times (2022, first 6 months)', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig('jitter_plot.png', dpi=150, bbox_inches='tight'); plt.show()
print('The clustering at round hours (0, 6, 12, 18) reveals that officers often record incident time as an approximation rather than the exact minute.')


In [ ]:
# ── Histogram of GPS latitude — two crimes compared ───────────────────
crimes_to_compare = ['Larceny Theft', 'Assault']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, crime in zip(axes, crimes_to_compare):
    subset = df_focus[(df_focus['category']==crime) &
                      df_focus['lat'].between(37.70, 37.83)]['lat']
    counts, bin_edges = np.histogram(subset, bins=50)
    bin_centers = 0.5*(bin_edges[:-1] + bin_edges[1:])
    ax.bar(bin_centers, counts, width=bin_edges[1]-bin_edges[0],
           color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_xlabel('Latitude'); ax.set_ylabel('Count')
    ax.set_title(f'{crime} — Latitude Distribution', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('lat_histograms.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# ── KDE ──────────────────────────────────────────────────────────────
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(10, 4))
for crime, color in zip(['Larceny Theft','Assault'], ['steelblue','tomato']):
    lats = df_focus[(df_focus['category']==crime) &
                    df_focus['lat'].between(37.70, 37.83)]['lat'].dropna()
    kde = gaussian_kde(lats, bw_method='scott')
    x = np.linspace(37.70, 37.83, 400)
    ax.plot(x, kde(x), color=color, label=crime, linewidth=2)
ax.set_xlabel('Latitude'); ax.set_ylabel('Density')
ax.set_title('KDE of Latitude Distribution — Larceny Theft vs Assault', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('kde_lat.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# ── Probability plot (QQ plot) ────────────────────────────────────────
from scipy import stats as sp_stats

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, crime in zip(axes, ['Larceny Theft', 'Assault']):
    lats = df_focus[(df_focus['category']==crime) &
                    df_focus['lat'].between(37.70, 37.83)]['lat'].dropna()
    sp_stats.probplot(lats, dist='norm', plot=ax)
    ax.set_title(f'Probability Plot — {crime}', fontweight='bold')
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('probplot.png', dpi=150, bbox_inches='tight'); plt.show()
print('The reference distribution is the standard normal. Points on the line → normally distributed.')
print('Deviation at the tails → the distribution has heavier tails or is skewed.')
print('For crime, we expect strong deviations because crime concentrates in specific latitudes (hotspots).')


In [ ]:
# ── Box plots — crimes per day ───────────────────────────────────────
daily_counts = df_focus.groupby(['date','category']).size().unstack(fill_value=0)
# Re-index to ensure all focus crimes present
daily_counts = daily_counts.reindex(columns=FOCUS_CRIMES, fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
ax.boxplot([daily_counts[c].values for c in FOCUS_CRIMES],
           labels=FOCUS_CRIMES, vert=True, patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_xticklabels(FOCUS_CRIMES, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Incidents per day')
ax.set_title('Distribution of Daily Crime Counts by Category', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('boxplots_daily.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# ── Box plots — time of day ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
data_tod = []
for crime in FOCUS_CRIMES:
    hours = df_focus[(df_focus['category']==crime) &
                     df_focus['hour'].notna()]['hour'].values
    data_tod.append(hours)

ax.boxplot(data_tod, labels=FOCUS_CRIMES, vert=True, patch_artist=True,
           boxprops=dict(facecolor='darkorange', alpha=0.6))
ax.set_xticklabels(FOCUS_CRIMES, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Hour of day (0–23)')
ax.set_title('Time-of-Day Distribution by Crime Type', fontweight='bold')
ax.set_yticks(range(0, 25, 4))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('boxplots_tod.png', dpi=150, bbox_inches='tight'); plt.show()
print('Prostitution and Drug Offense peak late at night — the box plot wraps poorly around midnight,')
print('making night-time crimes appear to span the full day. A circular/polar representation would be better.')
